# Trading bot

In [ ]:
from ib_insync import *
import pandas as pd
import numpy as np
import datetime
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_curve, auc, roc_auc_score, precision_score, recall_score, confusion_matrix
import time
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
import ta

util.startLoop()  # solo en Jupyter / entornos con event loop activo

## Fetch data

In [28]:
def fetch_ibkr_history(symbol_str, timeframe='1 hour', years=10):
    """
    Descarga histórico de IBKR en bloques para evitar violaciones de pacing.
    symbol_str: 'SPY' o 'SPXL'
    """
    ib = IB()
    try:
        ib.connect('127.0.0.1', 4002, clientId=2) # Ajusta puerto/ID según TWS
    except:
        print("No se pudo conectar a TWS. Asegúrate de que está abierto.")
        return None

    contract = Stock(symbol_str, 'SMART', 'USD')
    
    # Definimos fechas
    end_time = datetime.datetime.now()
    all_bars = []
    
    # Iteramos hacia atrás en bloques de 1 año (seguro para velas de 1h)
    for _ in range(years):
        print(f"Descargando bloque terminando en: {end_time}")
        
        bars = ib.reqHistoricalData(
            contract,
            endDateTime=end_time,
            durationStr='1 Y',
            barSizeSetting=timeframe,
            whatToShow='TRADES',
            useRTH=True,  # Regular Trading Hours (Importante para evitar ruido nocturno)
            formatDate=1
        )
        
        if not bars:
            break
            
        all_bars.extend(reversed(bars)) # Guardamos invertido para luego ordenar
        
        # Actualizamos la fecha de fin al comienzo del bloque actual
        end_time = bars[0].date
        time.sleep(2) # Pausa para respetar la API
        
    df = util.df(list(reversed(all_bars))) # Convertimos a DataFrame
    ib.disconnect()
    
    # Limpieza básica
    if df is not None:
        df['date'] = pd.to_datetime(df['date'])
        df.set_index('date', inplace=True)
        df = df[~df.index.duplicated(keep='first')] # Eliminar duplicados por solapamiento
    
    return df

# Ejemplo de uso:
# df_spy = fetch_ibkr_history('SPY', years=10)
df_spxl = fetch_ibkr_history('SPXL', years=10)

Descargando bloque terminando en: 2026-02-06 17:07:21.887106
Descargando bloque terminando en: 2025-02-07 09:30:00-05:00
Descargando bloque terminando en: 2024-02-08 09:30:00-05:00
Descargando bloque terminando en: 2023-02-08 09:30:00-05:00
Descargando bloque terminando en: 2022-02-08 09:30:00-05:00
Descargando bloque terminando en: 2021-02-08 09:30:00-05:00
Descargando bloque terminando en: 2020-02-07 09:30:00-05:00
Descargando bloque terminando en: 2019-02-07 09:30:00-05:00
Descargando bloque terminando en: 2018-02-07 09:30:00-05:00
Descargando bloque terminando en: 2017-02-07 09:30:00-05:00


## Feature engineering

In [103]:
def create_features(df_input):
    """
    Crea un set de features básico y robusto para empezar.
    Solo 4 variables clave: Tendencia, Momentum, Volatilidad y Volumen.
    """
    df = df_input.copy()
    
    # 1. TENDENCIA: Distancia a la media de 200 (El "Filtro de Régimen")
    # Si es > 0, estamos en tendencia alcista. Si es < 0, bajista.
    df['SMA_200'] = ta.trend.sma_indicator(df['close'], window=200)
    df['Dist_SMA200'] = (df['close'] / df['SMA_200']) - 1
    
    # 2. MOMENTUM: RSI (El clásico)
    # Nos dice si está sobrecomprado (>70) o sobrevendido (<30).
    df['RSI'] = ta.momentum.rsi(df['close'], window=14)
    
    # 3. VOLATILIDAD: ATR Porcentual
    # Necesario para saber cuánto se mueve el precio hoy comparado con ayer.
    df['ATR'] = ta.volatility.average_true_range(df['high'], df['low'], df['close'], window=14)
    df['ATR_Pct'] = df['ATR'] / df['close']
    
    # 4. VOLUMEN: Volumen Relativo Simple
    # ¿Hay más volumen hoy que la media del último mes? (Ratio > 1 es mucho volumen)
    df['Vol_Mean'] = df['volume'].rolling(20).mean()
    df['Vol_Rel'] = df['volume'] / df['Vol_Mean']
    
    # Limpieza de nulos (las primeras 200 filas serán NaN por la SMA_200)
    df.dropna(inplace=True)
    
    # Definimos cuáles son las columnas que usará el modelo (X)
    features_list = ['Dist_SMA200', 'RSI', 'ATR_Pct', 'Vol_Rel']
    
    return df, features_list

In [118]:
import pandas as pd
import numpy as np
import ta

def create_features(df_input):
    """
    Features avanzadas enfocadas en Mean Reversion y Flujo de Dinero.
    """
    df = df_input.copy()
    
    # --- 1. TENDENCIA & FILTRO (Macro) ---
    # Distancia SMA 200 (Filtro de régimen largo)
    df['SMA_200'] = ta.trend.sma_indicator(df['close'], window=200)
    df['Dist_SMA200'] = (df['close'] / df['SMA_200']) - 1
    
    # Distancia EMA Corta (8 vs 21) - Detecta el swing inmediato
    df['EMA_8'] = ta.trend.ema_indicator(df['close'], window=8)
    df['EMA_21'] = ta.trend.ema_indicator(df['close'], window=21)
    # Oscilador de tendencia corto plazo
    df['Trend_Pull'] = (df['EMA_8'] - df['EMA_21']) / df['close']

    # --- 2. MOMENTUM & FLUJO (El motor) ---
    # RSI (Estático)
    df['RSI'] = ta.momentum.rsi(df['close'], window=14)
    
    # RSI Slope (Dinámico): ¿El RSI está acelerando?
    # Importante: Diferenciar un RSI 40 bajando (malo) de un RSI 40 subiendo (bueno)
    df['RSI_Slope'] = df['RSI'] - df['RSI'].shift(3)
    
    # MFI (Money Flow Index): RSI pero ponderado por Volumen. 
    # Detecta divergencias mejor que el RSI simple.
    df['MFI'] = ta.volume.money_flow_index(df['high'], df['low'], df['close'], df['volume'], window=14)

    # --- 3. VOLATILIDAD & MEAN REVERSION (Bandas) ---
    # Bollinger Bands %B: ¿Dónde está el precio dentro de las bandas?
    # < 0: Rompió abajo (Sobrevendido extremo) | > 1: Rompió arriba
    bb = ta.volatility.BollingerBands(df['close'], window=20, window_dev=2)
    df['BB_PctB'] = bb.bollinger_pband()
    
    # BB Width (Squeeze): ¿Se están estrechando las bandas? (Precursor de explosión)
    df['BB_Width'] = bb.bollinger_wband()
    
    # ATR Relativo (ya lo tenías, es bueno mantenerlo)
    df['ATR'] = ta.volatility.average_true_range(df['high'], df['low'], df['close'], window=14)
    df['ATR_Pct'] = df['ATR'] / df['close']

    # --- 4. ESTACIONALIDAD (Time Features) ---
    # El SPY tiene patrones claros según la hora
    # Convertimos hora cíclica (seno/coseno) para que 23:00 esté cerca de 00:00
    if isinstance(df.index, pd.DatetimeIndex):
        df['Hour_Sin'] = np.sin(2 * np.pi * df.index.hour / 24)
        df['Hour_Cos'] = np.cos(2 * np.pi * df.index.hour / 24)
    
    # --- LIMPIEZA ---
    df.dropna(inplace=True)
    
    # Lista actualizada de features
    features_list = [
        'Dist_SMA200', 'Trend_Pull',     # Tendencia
        'RSI', 'RSI_Slope', 'MFI',       # Momentum/Volumen
        'BB_PctB', 'BB_Width', 'ATR_Pct',# Volatilidad
        'Hour_Sin', 'Hour_Cos'           # Tiempo
    ]
    
    return df, features_list

## Target definition

In [119]:
def get_triple_barrier_label(df, horizon=24, sl_mult=2.0, tp_mult=3.0):
    """
    Etiqueta usando ATR Porcentual (ATR_Pct).
    
    Lógica:
    - Si el precio sube X veces la volatilidad % -> 1
    - Si baja Y veces la volatilidad % -> 0
    - Si se acaba el tiempo -> 0
    """
    
    # 1. Verificación de seguridad
    if 'ATR_Pct' not in df.columns:
        raise ValueError("El DataFrame necesita la columna 'ATR_Pct'")

    # Usamos la columna porcentual
    volatility_pct = df['ATR_Pct']
    
    labels = []
    
    # Convertimos a numpy arrays para velocidad en lectura
    closes = df['close'].values
    vols_pct = volatility_pct.values
    
    # Iteramos
    for i in range(len(df) - horizon):
        curr_price = closes[i]
        curr_vol_pct = vols_pct[i]
        
        # --- CAMBIO IMPORTANTE: CÁLCULO MULTIPLICATIVO ---
        # Antes: curr_price - (curr_vol_absoluta * sl_mult)
        # Ahora: curr_price * (1 - (curr_vol_pct * sl_mult))
        
        stop_loss = curr_price * (1 - (curr_vol_pct * sl_mult))
        take_profit = curr_price * (1 + (curr_vol_pct * tp_mult))
        
        # Miramos hacia el futuro 'horizon' velas
        future_window = closes[i+1 : i+1+horizon]
        
        # Chequeo: ¿Tocó alguna barrera?
        # np.any es muy rápido para verificar existencia
        hit_tp = np.any(future_window >= take_profit)
        hit_sl = np.any(future_window <= stop_loss)
        
        if hit_tp and not hit_sl:
            # Solo tocó TP
            labels.append(1)
            
        elif hit_tp and hit_sl:
            # Tocó ambos, hay que ver cuál fue primero
            # argmax devuelve el índice del primer True
            idx_tp = np.argmax(future_window >= take_profit)
            idx_sl = np.argmax(future_window <= stop_loss)
            
            if idx_tp < idx_sl:
                labels.append(1) # Ganó el TP
            else:
                labels.append(0) # Saltó el SL antes
        else:
            # No tocó nada o solo tocó SL
            labels.append(0)
            
    # Rellenamos los últimos valores (donde no hay futuro suficiente) con NaN
    # Importante: extendemos la lista para que coincida con el len(df) original
    labels = labels + [np.nan] * horizon
    
    df['Target'] = labels
    
    # Devolvemos el DF limpio, eliminando las filas finales que no tienen Target
    return df.dropna(subset=['Target'])

## Training

In [120]:
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit

def train_model(df, features_to_keep):
    X = df[features_to_keep]
    y = df['Target']
    
    # Split temporal simple (último 20% para test)
    split_point = int(len(df) * 0.8)
    X_train, X_test = X.iloc[:split_point], X.iloc[split_point:]
    y_train, y_test = y.iloc[:split_point], y.iloc[split_point:]
    
    # Configuración del modelo (Robusta para evitar overfitting)
    model = xgb.XGBClassifier(
    n_estimators=200,        # Más árboles
    max_depth=2,             # Árboles MUY pequeños (evita memorizar ruido)
    learning_rate=0.05,      # Aprendizaje lento
    subsample=0.5,           # Usa solo la mitad de los datos por árbol (reduce varianza)
    colsample_bytree=0.5,    # Usa solo la mitad de features por árbol
    gamma=0.5,               # <--- CLAVE: Umbral mínimo para crear una rama
    reg_alpha=0.1,           # Regularización L1 (Lasso)
    reg_lambda=1.0,          # Regularización L2 (Ridge)
    scale_pos_weight=1,      # Ajustar según ratio (ej: Negativos / Positivos)
    eval_metric='auc'
    )
    
    model.fit(X_train, y_train)
    
    # Predicciones
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1] # Probabilidad de clase 1
    
    print(f"Precision Score (Test): {precision_score(y_test, preds):.2f}")
    
    return model, X_test, y_test, probs

In [121]:
df_spxl_features, features_to_keep = create_features(df_spxl)
df_spxl_features_and_target = get_triple_barrier_label(df_spxl_features)

print(df_spxl_features_and_target.Target.value_counts(normalize=True))
print("\n")

model, X_test, y_test, probs = train_model(df_spxl_features_and_target, features_to_keep)

Target
0.0    0.646298
1.0    0.353702
Name: proportion, dtype: float64


Precision Score (Test): 0.57


## Model evaluation

In [ ]:
def evaluate_trading_model(model, X_test, y_test, prob_threshold=0.6):
    """
    Evalúa el modelo con un umbral de decisión específico.
    Retorna un diccionario con las métricas clave.
    """
    
    # 1. Obtener probabilidades (Clase 1 = Compra)
    probs = model.predict_proba(X_test)[:, 1]
    
    # 2. Calcular AUC (Independiente del umbral, mide la calidad global)
    try:
        auc = roc_auc_score(y_test, probs)
    except:
        auc = 0.5 # Si falla (ej: solo hay una clase en test), asumimos random
        
    # 3. Aplicar el Umbral (Convertir probabilidad en señal 0 o 1)
    # Importante: No usamos model.predict() porque eso usa umbral 0.5 por defecto
    predictions = (probs > prob_threshold).astype(int)
    
    # 4. Métricas de Trading
    n_signals = predictions.sum()
    total_samples = len(y_test)
    
    if n_signals > 0:
        win_rate = precision_score(y_test, predictions) # % de veces que acertamos al comprar
    else:
        win_rate = 0.0
        
    # 5. Desglose de Aciertos/Fallos (Confusion Matrix)
    # TN: No compramos y bajó (Bien) | FP: Compramos y bajó (MAL - Pérdida)
    # FN: No compramos y subió (Oportunidad perdida) | TP: Compramos y subió (BIEN - Ganancia)
    tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()
    
    # --- REPORTE IMPRESO ---
    print(f"========== EVALUACIÓN MODELO (Umbral > {prob_threshold}) ==========")
    print(f"1. CALIDAD GLOBAL (AUC):      {auc:.2%} {'(Bueno)' if auc > 0.55 else '(Ruido)'}")
    print(f"2. ACTIVIDAD:")
    print(f"   - Señales Emitidas:        {n_signals} de {total_samples} velas ({n_signals/total_samples:.1%})")
    print(f"3. RENDIMIENTO (Win Rate):    {win_rate:.2%}  <-- DATO CLAVE")
    print(f"4. DETALLE OPERATIVO:")
    print(f"   - ✅ Aciertos (TP):        {tp}")
    print(f"   - ❌ Fallos (FP):          {fp} (Stop Loss)")
    print(f"   - 💤 Oportunidades (FN):   {fn} (Señales que perdimos)")
    print("===============================================================")
    
    return {
        'AUC': auc,
        'Signals': n_signals,
        'Win_Rate': win_rate,
        'TP': tp, 
        'FP': fp,
        'Probs': probs # Devolvemos las probs por si quieres graficar luego
    }

# --- EJEMPLO DE USO ---
metrics = evaluate_trading_model(model, X_test, y_test, prob_threshold=0.5)

========== EVALUACIÓN MODELO (Umbral > 0.5) ==========
1. CALIDAD GLOBAL (AUC):      53.53% (Ruido)
2. ACTIVIDAD:
   - Señales Emitidas:        21 de 3463 velas (0.6%)
3. RENDIMIENTO (Win Rate):    57.14%  <-- DATO CLAVE
4. DETALLE OPERATIVO:
   - ✅ Aciertos (TP):        12
   - ❌ Fallos (FP):          9 (Stop Loss)
   - 💤 Oportunidades (FN):   1150 (Señales que perdimos)


In [123]:
import plotly.graph_objects as go

def plot_backtest(df_test, y_test, probs, threshold=0.6):
    
    # Filtramos solo donde el modelo hubiera comprado
    buy_signals = df_test[probs > threshold].index
    
    # Gráfico de Velas
    fig = go.Figure(data=[go.Candlestick(x=df_test.index,
                    open=df_test['open'],
                    high=df_test['high'],
                    low=df_test['low'],
                    close=df_test['close'],
                    name='SPY')])

    # Añadir Señales de Compra
    # Verde si fue un acierto (Target=1), Rojo si fue fallo (Target=0)
    
    # Recuperamos el target real para colorear
    actual_results = y_test.loc[buy_signals]
    colors = ['green' if x == 1 else 'red' for x in actual_results]
    
    fig.add_trace(go.Scatter(
        x=buy_signals, 
        y=df_test.loc[buy_signals]['low'] * 0.99, # Dibujar un poco por debajo
        mode='markers',
        marker=dict(symbol='triangle-up', size=12, color=colors),
        name='Señal Modelo'
    ))

    fig.update_layout(
        title='Backtest Visual: Triángulos Verdes (Acierto) vs Rojos (Fallo)',
        yaxis_title='Precio SPY',
        xaxis_rangeslider_visible=False,
        template='plotly_dark'
    )
    
    fig.show()

plot_backtest(X_test.join(df_spxl), y_test, probs, threshold=0.55)

## BUY / SELL strategy

In [124]:
def get_actionable_signal(model, current_data_row, probability_threshold=0.60):
    """
    Retorna la decisión basada en la probabilidad del modelo.
    threshold: 0.60 (Exigimos un 60% de certeza, no un 50%)
    """
    # Asumimos que current_data_row ya tiene las features calculadas
    features_val = current_data_row.values.reshape(1, -1)
    
    # Predecir probabilidad
    prob_up = model.predict_proba(features_val)[0][1]
    
    decision = "NEUTRAL"
    if prob_up > probability_threshold:
        decision = "COMPRA FUERTE (BULL)"
    elif prob_up < 0.4:
        decision = "EVITAR / VENTA"
        
    return decision, prob_up